Refactored compute_correlations script.

[Runtime: <1 min total per scan file]

---------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, subprocess
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import FastICA
from scipy.stats import pearsonr, skew
from sklearn.covariance import LedoitWolf
import networkx as nx
from scipy.stats import skew

In [ ]:
### LOAD FREESURFER:
FREESURFER_HOME = config['freesurfer']['home']
FREESURFER_LICENSE = Path(config['freesurfer'].get('license'))
FREESURFER_SUBJECTS_DIR = config['freesurfer']['subjects_dir']
os.environ['FREESURFER_HOME'] = str(FREESURFER_HOME)
os.environ['SUBJECTS_DIR']    = str(FREESURFER_SUBJECTS_DIR)
os.environ['PATH'] = f"{str(FREESURFER_HOME)}/bin:" + os.environ.get('PATH', '')
if FREESURFER_LICENSE.is_file():
    os.environ['FS_LICENSE'] = str(FREESURFER_LICENSE)
else:
    raise RuntimeError("FreeSurfer license not found: check that 'license.txt' exists in home FreeSurfer directory."
    "in FreeSurfer $HOME directory and that a valid path is set in 'config.yaml' file.")
subprocess.run("recon-all --version", shell=True, check=True)

In [ ]:
# __________________________________________________________________________________________________________
### LOAD PARAMETERS:

# General parameters:
SUBSET       = config['subset']
HARD_STOP    = config['hard_errors']
RANDOM_SEED  = config['random_seed']

# Optional filtering params:
FILTER_SUBJECT_IDS = config["filter"].get("subject_IDs", [])
FILTER_SESSION_IDS = config["filter"].get("session_IDs", [])
FILTER_GROUP_IDS   = config["filter"].get("group_IDs", [])

# Correlation / graph parameters (NO validation here; upstream already enforces correctness):
compute_cfg = config['compute_correlations']

CORRELATION_TYPE = compute_cfg['correlation_type']          # 'pearson' or 'partial'

# centrality_types -- allow either 'centrality_type' (single) or 'centrality_types' (list):
centrality_raw = compute_cfg.get('centrality_type',
                 compute_cfg.get('centrality_types', []))
if isinstance(centrality_raw, str):
    CENTRALITY_TYPES = [centrality_raw]
else:
    CENTRALITY_TYPES = list(centrality_raw)

GRAPH_METRICS = compute_cfg.get('graph_metrics', [])
if isinstance(GRAPH_METRICS, str):
    GRAPH_METRICS = [GRAPH_METRICS]
else:
    GRAPH_METRICS = list(GRAPH_METRICS)

ABSOLUTE_WEIGHTS = compute_cfg.get(
    'use_absolute',
    compute_cfg.get('absolute_weight_values', True))

EPOCH_LENGTH = compute_cfg.get('epoch_length', None)
if EPOCH_LENGTH in (None, 0, 0.0):  # <-- Normalize null/zero to 'None' (no epoching)
    EPOCH_LENGTH = None
else:
    EPOCH_LENGTH = int(EPOCH_LENGTH)

EPOCH_OVERLAP = compute_cfg.get('epoch_overlap', 0)
if EPOCH_OVERLAP in (None, 0, 0.0):  # <-- Normalize null/zero to 0 (no overlap)
    EPOCH_OVERLAP = 0
else:
    EPOCH_OVERLAP = int(EPOCH_OVERLAP)

if EPOCH_LENGTH is not None:
    assert EPOCH_LENGTH > EPOCH_OVERLAP, (
        "[ERROR] Invalid epoching parameters: total window size ('epoch_length') "
        "must be larger than window overlap ('epoch_overlap'); revise CONFIG file.")

MANUAL_TR = compute_cfg.get('manual_TR', None)

# Whether to overwrite graph-metrics outputs (used in export cell):
OVERWRITE_METRICS = bool(compute_cfg.get('overwrite_metrics', False))

# From parcellation block -- ensure uniform length of time series across subjects?:
ENSURE_UNIFORM_LENGTH = config['ensure_uniform_length']

# Parcellation parameters (NO additional checks needed; already validated upstream):
TIMESERIES_TYPE = str(config['parcellation']['timeseries_type']).strip()   # <-- e.g. 'mean' or 'norm'
TIMESERIES_TYPE_TOKEN = TIMESERIES_TYPE.upper()                            # 'MEAN' or 'NORM' for filenames

ATLAS_FAMILY        = config['parcellation']['atlas']                      # <-- 'Craddock' | 'Schaefer' | 'MIST'
NUM_ROIS            = config['parcellation']['n_rois']
ATLAS_NETWORK_SCALE = config['parcellation'].get('network_scale', None)

# Convenience -- for filename-building later:
ATLAS_FAMILY_LOWER = str(ATLAS_FAMILY).lower()

# __________________________________________________________________________________________________________
### SET FILEPATHS:

BASE_DIRECTORY = Path(config['root_output_directory'])

PARCELLATION_DIR = BASE_DIRECTORY / config['parcellation_output_dir']

RUN_MANIFEST_PATH    = BASE_DIRECTORY / 'subject_manifest.csv'
fMRI_PARAMETERS_PATH = BASE_DIRECTORY / 'fMRI_manifest.csv'

GRAPH_METRICS_OUTPUT_DIR = BASE_DIRECTORY / compute_cfg['metrics_output_dir']
GRAPH_METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# __________________________________________________________________________________________________________
### INITIALIZATION:

RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
fMRI_runs    = pd.read_csv(fMRI_PARAMETERS_PATH)

In [ ]:
# ======================================================================
# FILTERING (group_ID → session_ID → subject_ID) + DIAGNOSTIC SUBSETTING
# ======================================================================

# ---- FILTERING (if enabled) ----
any_filters_active = bool(FILTER_GROUP_IDS or FILTER_SESSION_IDS or FILTER_SUBJECT_IDS)
if any_filters_active:
    print("\n[FILTER] Applying YAML-defined filters to fMRI_runs...")
    print(f"[FILTER] Starting with {len(fMRI_runs):,} rows.")
    # 1) Filter by group_IDs (substrings, case-insensitive):
    if FILTER_GROUP_IDS:
        n_before = len(fMRI_runs)
        pattern = "|".join(re.escape(val) for val in FILTER_GROUP_IDS)
        mask = fMRI_runs["group_ID"].astype(str).str.contains(pattern, case=False, na=False)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] group_IDs {FILTER_GROUP_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 2) Filter by session_IDs (exact matches):
    if FILTER_SESSION_IDS:
        n_before = len(fMRI_runs)
        mask = fMRI_runs["session_ID"].astype(str).isin(FILTER_SESSION_IDS)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] session_IDs {FILTER_SESSION_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 3) Filter by subject_IDs (exact matches):
    if FILTER_SUBJECT_IDS:
        n_before = len(fMRI_runs)
        mask = fMRI_runs["subject_ID"].astype(str).isin(FILTER_SUBJECT_IDS)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] subject_IDs {FILTER_SUBJECT_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    print(f"[FILTER] Final row count after all filters: {len(fMRI_runs):,} rows.\n")

# ---- DIAGNOSTIC SUBSETTING (if enabled) ----
if isinstance(SUBSET, int) and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Running only {SUBSET} test subjects/files:")
    fMRI_runs = fMRI_runs.head(SUBSET).copy()

if any_filters_active or SUBSET:
    fMRI_runs = fMRI_runs.reset_index(drop=True)
    display(fMRI_runs)

Data check / audit:

In [ ]:
# __________________________________________________________________________________________________________
### Build filetargets_df: find each subject/session’s time-series CSV

required_columns = ["subject_ID", "session_ID"]
for column_name in required_columns:
    if column_name not in fMRI_runs.columns:
        raise RuntimeError(f"fMRI_runs is missing required column: '{column_name}'")

filetargets_rows = []

for row_index, row in fMRI_runs.iterrows():

    subject_id = str(row["subject_ID"])
    session_id = str(row["session_ID"])
    prefix = f"{subject_id}_{session_id}"

    # ------------------------------------------------------------------
    # Construct filename according to parcellation naming convention:
    #
    #   <prefix>_<TIMESERIES_TYPE_TOKEN>_<ATLAS_FAMILY>_<NUM_ROIS>.csv
    #   <prefix>_<TIMESERIES_TYPE_TOKEN>_<ATLAS_FAMILY>_<NUM_ROIS>_net<NETWORK>.csv     <-- For Schaefer atlas parcellations only
    # ------------------------------------------------------------------

    filename_parts = [
        prefix,
        TIMESERIES_TYPE_TOKEN,   # <-- 'MEAN' or 'NORM'
        ATLAS_FAMILY,
        str(NUM_ROIS)]

    if ATLAS_NETWORK_SCALE not in (None, "", "None"):
        filename_parts.append(f"net{ATLAS_NETWORK_SCALE}")

    timeseries_filename = "_".join(filename_parts) + ".csv"
    timeseries_filepath = PARCELLATION_DIR / prefix / timeseries_filename

    if not timeseries_filepath.is_file():
        message = (f"[MISSING] Expected parcellation time-series file not found for prefix={prefix}: "
                   f"{timeseries_filepath}")
        if HARD_STOP:
            raise FileNotFoundError(message)
        else:
            print(message)
            continue

    # Determine TR:
    if MANUAL_TR is not None:
        repetition_time = float(MANUAL_TR)
    else:
        if "RepetitionTime" not in fMRI_runs.columns:
            raise RuntimeError(
                "fMRI_runs missing 'RepetitionTime' and manual_TR is null — cannot determine TR.")
        repetition_time = row["RepetitionTime"]

    # Count time samples (columns t_0001..t_N):
    header_df = pd.read_csv(timeseries_filepath, nrows=0)
    num_cols = len(header_df.columns)
    if num_cols < 3:
        raise ValueError(f"{timeseries_filepath} has too few columns.")

    num_time_samples = num_cols - 2

    filetargets_rows.append({
        "subject_ID": subject_id,
        "session_ID": session_id,
        "prefix": prefix,
        "timeseries_filepath": str(timeseries_filepath),
        "num_total_samples": int(num_time_samples),
        "TR": float(repetition_time)})

filetargets_df = (
    pd.DataFrame(filetargets_rows)
      .sort_values(["subject_ID", "session_ID"])
      .reset_index(drop=True))

print(f"[FILES] Built filetargets_df with {len(filetargets_df)} rows:\n")
display(filetargets_df.head())

# __________________________________________________________________________________________________________
### Cohort-level consistency checks:

unique_counts = filetargets_df["num_total_samples"].unique()
print("\nSANITY CHECKS:")

# uniform-length check (ONLY if 'ENSURE_UNIFORM_LENGTH'==True):
if ENSURE_UNIFORM_LENGTH:
    if unique_counts.size != 1:
        print("[ERROR] Inconsistent number of samples across time-series files:")
        display(filetargets_df[["prefix", "timeseries_filepath", "num_total_samples"]])
        raise RuntimeError(
            "num_total_samples is not consistent across subjects and "
            "ensure_uniform_length=True — cannot proceed.")
    total_samples = int(unique_counts[0])
    print(f"   [OK] All time-series have {total_samples} samples.")

else:
    # No explicit length enforcement here — just report the range:
    min_samps = int(filetargets_df["num_total_samples"].min())
    max_samps = int(filetargets_df["num_total_samples"].max())
    if min_samps != max_samps:
        print(f"   [INFO] Time-series lengths vary across subjects "
              f"(min={min_samps}, max={max_samps}).")
    else:
        print(f"   [OK] All time-series have {min_samps} samples (uniform by coincidence).")

    total_samples = None  # <-- per-subject length will be used later

# ----------------------------------------------------------------------
# Epoching check
# ----------------------------------------------------------------------

if EPOCH_LENGTH is None:
    print("   [OK] Epoching disabled: correlations computed across entire recording.")

else:

    if ENSURE_UNIFORM_LENGTH:
        # Standard epoching rules based on a single common length:
        step_size = EPOCH_LENGTH - EPOCH_OVERLAP
        if step_size <= 0:
            raise RuntimeError(
                f"Invalid epoch settings: epoch_length={EPOCH_LENGTH}, "
                f"epoch_overlap={EPOCH_OVERLAP}")

        if total_samples < EPOCH_LENGTH:
            raise RuntimeError(
                f"epoch_length={EPOCH_LENGTH} > total_samples={total_samples}; no windows possible.")

        num_windows = 1 + (total_samples - EPOCH_LENGTH) // step_size
        remainder = total_samples - (EPOCH_LENGTH + (num_windows - 1) * step_size)

        if remainder > 0:
            print(f"   [WARN] Windows leave {remainder} trailing samples unused.")

        print(f"   [OK] epoch_length={EPOCH_LENGTH}, overlap={EPOCH_OVERLAP}, "
              f"step={step_size}, windows={num_windows}.")

    else:
        # If variable lengths allowed, windowing will happen per subject:
        print("   [INFO] epoch_length set, but ensure_uniform_length=False —")
        print("         → windowing will be computed individually per subject.")

-----------


In [ ]:
# __________________________________________________________________________________________________________
### Helper: compute centrality for a single time window

def compute_window_centrality(window_df,
                              centrality_type="eigenvector",
                              correlation_type="pearson",
                              use_absolute_weights=True,
                              threshold=0.0,
                              return_extras=False):
    """
    Computes a centrality vector for a windowed ROI time-series matrix.

    Parameters
    ----------
    window_df : pandas.DataFrame
        Shape (n_rois, n_timepoints), ideally z-scored over session (NORM files).
        Rows = ROI labels (ints), columns = consecutive timepoints (t_XXXX).
    centrality_type : str
        'eigenvector', 'degree', 'betweenness', 'closeness', or 'pagerank'.
    correlation_type : str
        'pearson' or 'partial' (graph built from partial corr via precision matrix).
    use_absolute_weights : bool
        If True, edge weights are |corr| before centrality is computed.
    threshold : float, default 0.0
        Minimum absolute correlation to keep an edge (0 means no threshold).
    return_extras : bool
        If True, also return extras dict with corr_matrix, graph, roi_labels.

    Returns
    -------
    centrality_vector : np.ndarray (length n_rois)
        Centrality per ROI for this window.
    (optionally extras dict if return_extras=True)
    """
    n_rois = window_df.shape[0]

    # Compute correlation matrix:
    if correlation_type == "pearson":
        # Enforce numeric (robust to CSV/object dtype issues):
        w = window_df.apply(pd.to_numeric, errors="coerce")
        w = w.replace([np.inf, -np.inf], np.nan)
        # Fill missing values minimally: row-wise means; then 0 if row is all-NaN:
        if w.isna().any().any():
            w = w.T.fillna(w.mean(axis=1)).T
            w = w.fillna(0.0)
        corr_matrix = w.T.corr(method="pearson").to_numpy(dtype=float)

    elif correlation_type == "partial":
        try:
            estimator = LedoitWolf()
            # NB: sklearn expects (n_samples, n_features) = (T, n_rois)
            estimator.fit(window_df.T)
            # Prefer estimator 'precision_' (numerically safer than explicit inverse):
            precision = getattr(estimator, "precision_", None)
            if precision is None:
                precision = np.linalg.inv(estimator.covariance_)
            precision = np.asarray(precision, dtype=float)
            # Guard against non-finite / degenerate diagonal:
            d = np.diag(precision).copy()
            if (not np.all(np.isfinite(d))) or np.any(d <= 0):
                raise ValueError("[WARNING] Precision diagonal non-finite or non-positive; cannot form partial correlations.")
            denom = np.sqrt(np.outer(d, d))
            corr_matrix = -precision / denom
            np.fill_diagonal(corr_matrix, 1.0)
        except Exception as e:
            # Fail loudly in the *values* (w/ NaNs), rather than silently returning zeros:
            corr_matrix = np.full((n_rois, n_rois), np.nan, dtype=float)

    else:
        raise ValueError("correlation_type must be 'pearson' or 'partial'")

    # Preprocess edges:
    np.fill_diagonal(corr_matrix, 0.0)

    if threshold > 0.0:
        corr_matrix[np.abs(corr_matrix) < threshold] = 0.0

    # Some centralities can’t handle signed/negative weights robustly;
    # If we compute closeness, force absolute weights to ensure non-negative distances:
    if centrality_type == "closeness":
        use_absolute_weights = True

    if use_absolute_weights:
        corr_matrix = np.abs(corr_matrix)

    # Build graph & compute centrality metrics:
    graph = nx.from_numpy_array(corr_matrix)

    if centrality_type == "eigenvector":
        centrality_dict = nx.eigenvector_centrality_numpy(graph, weight="weight")
    elif centrality_type == "degree":
        centrality_dict = dict(graph.degree(weight="weight"))
    elif centrality_type == "betweenness":
        centrality_dict = nx.betweenness_centrality(graph, weight="weight", normalized=True)
    elif centrality_type == "closeness":
        # Convert affinity weights -> distances (larger affinity = shorter distance):
        eps = 1e-12
        for u, v, d in graph.edges(data=True):
            w = float(d.get("weight", 0.0))
            if w > 0:
                d["distance"] = 1.0 / (w + eps)
            else:
                # If a non-positive edge somehow exists, treat as no-connection:
                d["distance"] = float("inf")
        centrality_dict = nx.closeness_centrality(graph, distance="distance")
    elif centrality_type == "pagerank":
        centrality_dict = nx.pagerank(graph, weight="weight")
    else:
        raise ValueError(
            "centrality_type must be one of: "
            "eigenvector, degree, betweenness, closeness, pagerank")

    centrality_vector = np.array(
        [centrality_dict[i] for i in range(n_rois)],
        dtype=float)

    if return_extras:
        extras = {
            "corr_matrix": corr_matrix,
            "graph": graph,
            "roi_labels": list(window_df.index)}
        return centrality_vector, extras

    return centrality_vector

-----------

Refactoring: Building one separate code section for processing the "un-epoched" data (always performed by default); then this will be followed by separate processing of an "epoched" branch so long as the 'epoch_length' config parameter is set to some valid value.

-----------

"Un-epoched", "whole-graph" branch:

In [ ]:
# __________________________________________________________________________________________________________
### UN-EPOCHED BRANCH: full-session centrality + whole-graph summaries (per subject/session)
#
# This cell:
#   - Loops over filetargets_df (one row per subject/session)
#   - Loads full-session ROI time-series from parcellation CSV
#   - Computes requested centrality metrics over the entire recording
#   - Computes whole-graph summaries (mean/variance/min/max/skew) across ROIs for each metric
#   - Optionally computes Q modularity for the full-session graph
#   - Stores:
#       CENTRALITY_UNEPOCHED[prefix]      -> ROI x centrality-metrics DataFrame
#       GLOBAL_UNEPOCHED_STATS[prefix]    -> 1-row DataFrame with summary stats per metric
#       GRAPH_Q_UNEPOCHED[prefix]         -> 1-row DataFrame with Q (if requested)
#
# NOTE: Epoching (EPOCH_LENGTH / EPOCH_OVERLAP) is **ignored** in this branch;
#       this always uses the full time-series for each subject.

CENTRALITY_UNEPOCHED   = {}  # key: prefix, value: DataFrame (index=ROI, columns=metrics)
GLOBAL_UNEPOCHED_STATS = {}  # key: prefix, value: 1-row DataFrame (whole-graph summaries)
GRAPH_Q_UNEPOCHED      = {}  # key: prefix, value: 1-row DataFrame with {'Q': value}

# Set whether to compute Q modularity at all (for any branch):
REQUEST_Q = any(str(metric_name).lower() == "q" for metric_name in GRAPH_METRICS)

if REQUEST_Q and not ABSOLUTE_WEIGHTS:
    print("[INFO] Q modularity requested with signed weights "
          "(absolute_weight_values/use_absolute=False). "
          "Results may be harder to interpret; consider enabling absolute weights for Q.")

for row_index, row in filetargets_df.iterrows():

    subject_id = str(row["subject_ID"])
    session_id = str(row["session_ID"])
    prefix     = str(row["prefix"])
    timeseries_filepath = str(row["timeseries_filepath"])
    num_total_samples   = int(row["num_total_samples"])

    print(f"\n[UN-EPOCHED] {prefix} | samples={num_total_samples}")

    # Load time-series CSV produced by the parcellation step; expects columns: ["ROI", "num_voxels", "t_0001", ...]:
    raw_df = pd.read_csv(timeseries_filepath)

    if raw_df.shape[1] < 3:
        print(f"[UN-EPOCHED SKIP] {prefix}: malformed timeseries CSV (columns={raw_df.shape[1]})")
        continue

    if "ROI" not in raw_df.columns or "num_voxels" not in raw_df.columns:
        print(f"[UN-EPOCHED SKIP] {prefix}: missing 'ROI' and/or 'num_voxels' columns.")
        continue

    # Sort by ROI & capture integer ROI labels:
    raw_df["ROI"] = raw_df["ROI"].astype(int)
    raw_df = raw_df.sort_values("ROI").reset_index(drop=True)
    roi_labels = raw_df["ROI"].tolist()
    num_rois   = len(roi_labels)

    # Extract the timeseries block (num_rois x num_timepoints):
    times_df = raw_df.iloc[:, 2:]
    if times_df.shape[1] != num_total_samples:
        print(
            f"[UN-EPOCHED WARN] {prefix}: header-counted num_total_samples={num_total_samples}, "
            f"but CSV has {times_df.shape[1]} time columns. Using CSV value.")
        num_total_samples = times_df.shape[1]

    # Use ROI labels as index:
    times_df.index = roi_labels

    # Full-session window = entire time-series:
    window_df = times_df  # <-- (num_rois x num_total_samples)

    # Compute centrality metrics for the full-session graph:
    centrality_vectors = {}
    graph_for_session  = None

    for centrality_type in CENTRALITY_TYPES:

        centrality_type_str = str(centrality_type).lower()

        # If Q modularity measures are requested, we get the graph once during the first centrality call:
        need_extras = REQUEST_Q and (graph_for_session is None)

        centrality_result = compute_window_centrality(
            window_df=window_df,
            centrality_type=centrality_type_str,
            correlation_type=CORRELATION_TYPE,
            use_absolute_weights=ABSOLUTE_WEIGHTS,
            threshold=0.0,
            return_extras=need_extras)

        if need_extras:
            centrality_vector, extras = centrality_result
            graph_for_session = extras["graph"]

            # Compute Q modularity once for the full-session graph:
            if REQUEST_Q:
                try:
                    communities = nx.algorithms.community.greedy_modularity_communities(
                        graph_for_session, weight="weight")
                    q_value = nx.algorithms.community.modularity(
                        graph_for_session, communities, weight="weight")
                except Exception:
                    q_value = np.nan
        else:
            centrality_vector = centrality_result

        base_name   = f"{centrality_type_str}Centrality"
        column_name = base_name  # <-- no '_time' suffix in un-epoched branch

        centrality_vectors[column_name] = centrality_vector

    if REQUEST_Q and graph_for_session is None:
        # FALLBACK: If Q was requested but we somehow never got a valid graph, mark as NaN:
        q_value = np.nan

    # Assemble centrality outputs into ROI x metrics DataFrame:
    if not centrality_vectors:
        print(f"[UN-EPOCHED WARN] {prefix}: no centrality vectors computed; skipping.")
        continue

    centrality_matrix = pd.DataFrame(
        centrality_vectors,
        index=roi_labels)

    # Diagnostic -- check for all-zero columns:
    zero_columns = centrality_matrix.columns[(centrality_matrix == 0).all()]
    if len(zero_columns) == 0:
        print(f"[UN-EPOCHED OK] {prefix}: centrality computed for all metrics (no all-zero columns).")
    else:
        print(f"[UN-EPOCHED WARN] {prefix}: {len(zero_columns)} all-zero centrality column(s): {list(zero_columns)}")

    print(f"[UN-EPOCHED SHAPE] centrality_matrix shape: {centrality_matrix.shape}")

    # Store ROI-level results for this branch:
    CENTRALITY_UNEPOCHED[prefix] = centrality_matrix.copy()

    # Whole-graph summaries across ROIs (mean/variance/min/max/skew per computed metric):
    summary_stats = {}

    for column_name in centrality_matrix.columns:
        values = centrality_matrix[column_name].to_numpy(dtype=float)

        # Basic distribution statistics across ROIs:
        summary_stats[f"{column_name}_mean"]      = float(np.mean(values))
        # Use sample variance (ddof=1) if >=2 ROIs, else 0.0:
        if values.size >= 2:
            summary_stats[f"{column_name}_variance"] = float(np.var(values, ddof=1))
        else:
            summary_stats[f"{column_name}_variance"] = 0.0
        summary_stats[f"{column_name}_min"]       = float(np.min(values))
        summary_stats[f"{column_name}_max"]       = float(np.max(values))

        # Skewness (handle degenerate/all-equal cases gracefully):
        try:
            skew_value = skew(values, bias=False)
            if not np.isfinite(skew_value):
                skew_value = 0.0
        except Exception:
            skew_value = 0.0
        summary_stats[f"{column_name}_skew"] = float(skew_value)

    # Attach Q (if requested) into the graph-level dict:
    if REQUEST_Q:
        summary_stats["Q"] = float(q_value)

    GLOBAL_UNEPOCHED_STATS[prefix] = pd.DataFrame([summary_stats])

    # Store Q modularity as a separate per-subject DataFrame (for export):
    if REQUEST_Q:
        GRAPH_Q_UNEPOCHED[prefix] = pd.DataFrame([{"Q": float(q_value)}])

print(f"\n[UN-EPOCHED DONE] Computed full-session centrality for {len(CENTRALITY_UNEPOCHED)} subject(s).")
if REQUEST_Q:
    print(f"[UN-EPOCHED DONE] Computed Q modularity for {len(GRAPH_Q_UNEPOCHED)} subject(s).")

-------------------

Next, the epoching-enabled branch:

In [ ]:
# __________________________________________________________________________________________________________
### EPOCHED BRANCH: windowed centrality + within-ROI summaries + Q summaries (per subject/session)
#
# This cell:
#   - Runs only if EPOCH_LENGTH is not None
#   - Loops over filetargets_df (one row per subject/session)
#   - Loads ROI time-series from parcellation CSV
#   - Builds sliding windows (epoch_length / epoch_overlap) in sample units
#   - For each window:
#       * Computes requested centrality metrics using compute_window_centrality(...)
#       * Optionally computes Q modularity for that window
#   - Assembles *tidy* outputs:
#       EPOCHED_TIMECOURSES[prefix] -> long DataFrame with columns:
#           ['subject_ID', 'session_ID', 'level', 'roi_label',
#            'metric_name', 'time_index', 'value']
#           - level = 'roi' for ROI-level centralities
#           - level = 'global' for whole-graph metrics (e.g. Q)
#       EPOCHED_SUMMARIES[prefix]   -> long DataFrame with columns:
#           ['subject_ID', 'session_ID', 'level', 'roi_label',
#            'metric_name', 'summary_type', 'value']
#           - summary_type ['mean', 'variance', 'min', 'max', 'skew', 'lag1_autocorr']
#           - level = 'roi' for ROI-level summaries
#           - level = 'global' for whole-graph summaries (e.g. Q)
#
# NOTE:
#   - This branch is independent from the un-epoched branch.
#   - Exporting & JSON-writing happens later in a dedicated export cell.
#   - Old wide-form structures (CENTRALITY_EPOCHED, SUMMARY_EPOCHED_ROI, GRAPH_Q_EPOCHED)
#     are replaced by these two tidy tables.

if EPOCH_LENGTH is None:
    print("\n[EPOCHED] EPOCH_LENGTH is None � epoched branch is disabled; skipping windowed analyses.")
else:
    # New tidy structures:
    EPOCHED_TIMECOURSES = {}  # key: prefix --> long DataFrame (time-resolved ROI + global metrics)
    EPOCHED_SUMMARIES   = {}  # key: prefix --> long DataFrame (summaries over windows)

    # Set whether to compute Q modularity at all:
    REQUEST_Q = any(str(metric_name).lower() == "q" for metric_name in GRAPH_METRICS)

    if REQUEST_Q and not ABSOLUTE_WEIGHTS:
        print("[EPOCHED INFO] Q modularity requested with signed weights "
              "(absolute_weight_values/use_absolute=False). "
              "Results may be harder to interpret; consider enabling absolute weights for Q.")

    # Helper: compute lag-1 autocorrelation for a 1D sequence:
    def lag1_autocorr(values_1d: np.ndarray) -> float:
        """
        Compute lag-1 Pearson autocorrelation for a 1D array.
        Returns 0.0 if fewer than 2 finite samples or if variance is zero.
        """
        arr = np.asarray(values_1d, dtype=float)
        # Keep only finite values
        finite_mask = np.isfinite(arr)
        arr = arr[finite_mask]
        if arr.size < 2:
            return 0.0
        x = arr[:-1]
        y = arr[1:]
        x_mean = x.mean()
        y_mean = y.mean()
        x_dev = x - x_mean
        y_dev = y - y_mean
        num = np.sum(x_dev * y_dev)
        den = np.sqrt(np.sum(x_dev ** 2) * np.sum(y_dev ** 2))
        if den == 0 or not np.isfinite(den):
            return 0.0
        return float(num / den)

    for row_index, row in filetargets_df.iterrows():

        subject_id = str(row["subject_ID"])
        session_id = str(row["session_ID"])
        prefix     = str(row["prefix"])
        timeseries_filepath = str(row["timeseries_filepath"])
        num_total_samples   = int(row["num_total_samples"])

        print(f"\n[EPOCHED] {prefix} | samples={num_total_samples}")

        # Load time-series CSV produced by the parcellation step; expects columns: ["ROI", "num_voxels", "t_0001", ...]:
        # ------------------------------------------------------------------
        raw_df = pd.read_csv(timeseries_filepath)

        if raw_df.shape[1] < 3:
            print(f"[EPOCHED SKIP] {prefix}: malformed timeseries CSV (columns={raw_df.shape[1]})")
            continue

        if "ROI" not in raw_df.columns or "num_voxels" not in raw_df.columns:
            print(f"[EPOCHED SKIP] {prefix}: missing 'ROI' and/or 'num_voxels' columns.")
            continue

        # Sort by ROI and capture integer ROI labels:
        raw_df["ROI"] = raw_df["ROI"].astype(int)
        raw_df = raw_df.sort_values("ROI").reset_index(drop=True)
        roi_labels = raw_df["ROI"].tolist()
        num_rois   = len(roi_labels)

        # Extract the timeseries block (num_rois x num_timepoints):
        times_df = raw_df.iloc[:, 2:]
        if times_df.shape[1] != num_total_samples:
            print(
                f"[EPOCHED WARN] {prefix}: header-counted num_total_samples={num_total_samples}, "
                f"but CSV has {times_df.shape[1]} time columns. Using CSV value.")
            num_total_samples = times_df.shape[1]

        # Use ROI labels as index:
        times_df.index = roi_labels

        # Determine windows for this subject (sliding or non-overlapping):
        #   - step_size = epoch_length - epoch_overlap
        #   - windows: [start, start+epoch_length), advancing by step_size
        step_size = EPOCH_LENGTH - EPOCH_OVERLAP
        if step_size <= 0:
            print(f"[EPOCHED ERROR] {prefix}: invalid epoch settings: "
                  f"epoch_length={EPOCH_LENGTH}, epoch_overlap={EPOCH_OVERLAP}")
            continue

        if num_total_samples < EPOCH_LENGTH:
            print(f"[EPOCHED WARN] {prefix}: num_total_samples={num_total_samples} < "
                  f"epoch_length={EPOCH_LENGTH}; skipping subject for windowed analysis.")
            continue

        window_specs = []
        window_index = 0
        start = 0
        while start + EPOCH_LENGTH <= num_total_samples:
            end = start + EPOCH_LENGTH
            window_specs.append((window_index, start, end))
            window_index += 1
            start += step_size

        if len(window_specs) == 0:
            print(f"[EPOCHED WARN] {prefix}: no valid windows formed; skipping.")
            continue

        num_windows = len(window_specs)
        print(f"[EPOCHED WINDOWS] num_windows={num_windows} "
              f"(epoch_length={EPOCH_LENGTH}, overlap={EPOCH_OVERLAP}, step={step_size})")

        # Compute centrality for each metric and each window:
        #   - centrality_vectors: dict[colname] -> np.ndarray (len num_rois)
        #   - q_values: dict[window_idx] -> float (Q modularity), if requested
        centrality_vectors = {}
        q_values = {}

        for window_index, start, end in window_specs:

            window_df = times_df.iloc[:, start:end]  # <-- shape: (num_rois, window_len)

            # For each window, we may compute the graph once (if Q is requested):
            graph_for_window = None

            for centrality_type in CENTRALITY_TYPES:

                centrality_type_str = str(centrality_type).lower()
                base_name = f"{centrality_type_str}Centrality"

                # Determine if we need extras (graph/corr_matrix) for this call:
                need_extras = REQUEST_Q and (graph_for_window is None)

                centrality_result = compute_window_centrality(
                    window_df=window_df,
                    centrality_type=centrality_type_str,
                    correlation_type=CORRELATION_TYPE,
                    use_absolute_weights=ABSOLUTE_WEIGHTS,
                    threshold=0.0,
                    return_extras=need_extras)

                if need_extras:
                    centrality_vector, extras = centrality_result
                    graph_for_window = extras["graph"]

                    # Compute Q modularity once per window using the current graph:
                    if REQUEST_Q:
                        try:
                            communities = nx.algorithms.community.greedy_modularity_communities(
                                graph_for_window, weight="weight")
                            q_value = nx.algorithms.community.modularity(
                                graph_for_window, communities, weight="weight")
                        except Exception:
                            q_value = np.nan
                        q_values[window_index] = float(q_value)
                else:
                    centrality_vector = centrality_result

                column_name = f"{base_name}_time{window_index + 1:03d}"
                centrality_vectors[column_name] = centrality_vector

            # FALLBACK -- If Q was requested but for some reason we never got a graph, mark this window as NaN:
            if REQUEST_Q and (window_index not in q_values):
                q_values[window_index] = np.nan

        # Assemble centrality outputs into ROI x (metric_timeXXX) DataFrame (intermediate), then convert to a tidy long table for timecourses:
        if not centrality_vectors and not REQUEST_Q:
            print(f"[EPOCHED WARN] {prefix}: no centrality vectors computed; skipping.")
            continue

        if centrality_vectors:
            centrality_matrix = pd.DataFrame(
                centrality_vectors,
                index=roi_labels)
        else:
            # No centrality metrics, but Q might be requested (edge-case):
            centrality_matrix = pd.DataFrame(index=roi_labels)

        # Diagnostic -- check for all-zero columns:
        if not centrality_matrix.empty:
            zero_columns = centrality_matrix.columns[(centrality_matrix == 0).all()]
            if len(zero_columns) == 0:
                print(f"[EPOCHED OK] {prefix}: centrality computed for all windows/metrics (no all-zero columns).")
            else:
                print(f"[EPOCHED WARN] {prefix}: {len(zero_columns)} all-zero centrality column(s): {list(zero_columns)}")

            print(f"[EPOCHED SHAPE] centrality_matrix shape: {centrality_matrix.shape}")
        else:
            print(f"[EPOCHED INFO] {prefix}: centrality_matrix is empty (no centrality metrics).")

        # Build tidy long table for time-resolved data:
        #   - ROI-level centrality metrics
        #   - Global Q per window (if requested)
        timecourse_rows = []

        # ROI-level centrality metrics:
        for centrality_type in CENTRALITY_TYPES:
            centrality_type_str = str(centrality_type).lower()
            base_name = f"{centrality_type_str}Centrality"

            metric_columns = [col for col in centrality_matrix.columns
                              if col.startswith(base_name + "_time")]

            for col in metric_columns:
                # Parse time index from suffix "..._timeXXX":
                try:
                    time_str = col.split("_time")[-1]
                    time_index = int(time_str)
                except Exception:
                    # If parsing fails, skip this column:
                    continue

                values = centrality_matrix[col].to_numpy(dtype=float)
                for roi_label, value in zip(roi_labels, values):
                    timecourse_rows.append({
                        "subject_ID": subject_id,
                        "session_ID": session_id,
                        "level": "roi",
                        "roi_label": int(roi_label),
                        "metric_name": base_name,
                        "time_index": time_index,
                        "value": float(value)})

        # Compute global Q (time-resolved):
        if REQUEST_Q:
            for win_idx in range(num_windows):
                time_index = win_idx + 1
                q_val = float(q_values.get(win_idx, np.nan))
                timecourse_rows.append({
                    "subject_ID": subject_id,
                    "session_ID": session_id,
                    "level": "global",
                    "roi_label": np.nan,
                    "metric_name": "Q",
                    "time_index": time_index,
                    "value": q_val})

        if not timecourse_rows:
            print(f"[EPOCHED WARN] {prefix}: no time-resolved rows assembled; skipping.")
            continue

        timecourses_df = pd.DataFrame(timecourse_rows)
        EPOCHED_TIMECOURSES[prefix] = timecourses_df

        # Compute within-ROI summaries across windows:
        summary_rows = []

        for centrality_type in CENTRALITY_TYPES:
            centrality_type_str = str(centrality_type).lower()
            base_name = f"{centrality_type_str}Centrality"

            metric_columns = [col for col in centrality_matrix.columns
                              if col.startswith(base_name + "_time")]

            if not metric_columns:
                continue

            values_2d = centrality_matrix[metric_columns].to_numpy(dtype=float)  # <-- (num_rois, num_windows_for_metric)
            num_windows_for_metric = values_2d.shape[1]
            num_rois_for_metric   = values_2d.shape[0]

            # Compute mean across windows:
            mean_vals = np.mean(values_2d, axis=1)
            # Sample variance if >=2 windows, else 0.0:
            if num_windows_for_metric >= 2:
                var_vals = np.var(values_2d, axis=1, ddof=1)
            else:
                var_vals = np.zeros_like(mean_vals)
            min_vals  = np.min(values_2d, axis=1)
            max_vals  = np.max(values_2d, axis=1)

            # Compute skewness across windows, per ROI:
            try:
                skew_vals = skew(values_2d, axis=1, bias=False)
                skew_vals[~np.isfinite(skew_vals)] = 0.0
            except Exception:
                skew_vals = np.zeros_like(mean_vals)

            # Compute lag-1 autocorrelation across windows, per ROI:
            lag1_vals = np.zeros(num_rois_for_metric, dtype=float)
            for i_roi in range(num_rois_for_metric):
                lag1_vals[i_roi] = lag1_autocorr(values_2d[i_roi, :])

            summary_types = ["mean", "variance", "min", "max", "skew", "lag1_autocorr"]
            arrays_by_summary = {
                "mean": mean_vals,
                "variance": var_vals,
                "min": min_vals,
                "max": max_vals,
                "skew": skew_vals,
                "lag1_autocorr": lag1_vals}

            for i_roi, roi_label in enumerate(roi_labels):
                for stype in summary_types:
                    value = float(arrays_by_summary[stype][i_roi])
                    summary_rows.append({
                        "subject_ID": subject_id,
                        "session_ID": session_id,
                        "level": "roi",
                        "roi_label": int(roi_label),
                        "metric_name": base_name,
                        "summary_type": stype,
                        "value": value})

        # Compute Q modularity summaries across windows (global):
        if REQUEST_Q:
            q_vector = np.array(
                [q_values.get(win_idx, np.nan) for win_idx in range(num_windows)],
                dtype=float)

            finite_mask = np.isfinite(q_vector)
            finite_q = q_vector[finite_mask]

            if finite_q.size == 0:
                q_mean = np.nan
                q_var  = np.nan
                q_min  = np.nan
                q_max  = np.nan
                q_skew = 0.0
                q_lag1 = 0.0
            else:
                q_mean = float(np.mean(finite_q))
                if finite_q.size >= 2:
                    q_var = float(np.var(finite_q, ddof=1))
                else:
                    q_var = 0.0
                q_min  = float(np.min(finite_q))
                q_max  = float(np.max(finite_q))
                try:
                    q_skew_arr = skew(finite_q, bias=False)
                    q_skew = float(q_skew_arr) if np.isfinite(q_skew_arr) else 0.0
                except Exception:
                    q_skew = 0.0
                q_lag1 = lag1_autocorr(finite_q)

            q_summary_values = {
                "mean": q_mean,
                "variance": q_var,
                "min": q_min,
                "max": q_max,
                "skew": q_skew,
                "lag1_autocorr": q_lag1}
            for stype, value in q_summary_values.items():
                summary_rows.append({
                    "subject_ID": subject_id,
                    "session_ID": session_id,
                    "level": "global",
                    "roi_label": np.nan,
                    "metric_name": "Q",
                    "summary_type": stype,
                    "value": float(value)})

        if summary_rows:
            summaries_df = pd.DataFrame(summary_rows)
            EPOCHED_SUMMARIES[prefix] = summaries_df
        else:
            print(f"[EPOCHED WARN] {prefix}: no summary rows assembled.")

    print(f"\n[EPOCHED DONE] Computed windowed metrics (tidy) for {len(EPOCHED_TIMECOURSES)} subject(s).")

--------
#### Save / export:

In [ ]:
# __________________________________________________________________________________________________________
### EXPORT CELL: write per-subject CSVs and JSON sidecars for un-epoched and epoched branches
#
# Behavior:
#   - Uses CENTRALITY_UNEPOCHED / GLOBAL_UNEPOCHED_STATS / GRAPH_Q_UNEPOCHED
#   - Uses EPOCHED_TIMECOURSES / EPOCHED_SUMMARIES (if EPOCH_LENGTH is not None)
#
#   Un-epoched outputs (per subject/session):
#       - <prefix>_un-epoched_centrality.csv
#           * Rows: ROIs (index = <ATLAS_FAMILY>_ROI_label)
#           * Cols: centrality metrics (e.g. eigenvectorCentrality, degreeCentrality, ...)
#       - <prefix>_un-epoched_global_metrics.csv
#           * Single row with:
#               - 'subject_ID'
#               - whole-graph summaries across ROIs:
#                   <metric>_mean, <metric>_variance, <metric>_min, <metric>_max, <metric>_skew
#               - 'Q' if requested
#
#   Epoched outputs (per subject/session; tidy format):
#       - <prefix>_epoched_window-<L>_overlap-<O>_timecourses.csv
#           Columns:
#               ['subject_ID', 'session_ID', 'level', 'roi_label',
#                'metric_name', 'time_index', 'value']
#           where:
#               level = 'roi'    → ROI-level centrality metrics (roi_label = int)
#               level = 'global' → global metrics (e.g. Q; roi_label = NaN)
#
#       - <prefix>_epoched_window-<L>_overlap-<O>_summaries.csv
#           Columns:
#               ['subject_ID', 'session_ID', 'level', 'roi_label',
#                'metric_name', 'summary_type', 'value']
#           where:
#               summary_type ∈ ['mean', 'variance', 'min', 'max', 'skew', 'lag1_autocorr']
#               level = 'roi'    → summaries across windows per ROI
#               level = 'global' → summaries across windows for global metrics (e.g. Q)
#
#   In all cases: JSON sidecars with detailed provenance:
#       - <prefix>_un-epoched_provenance.json
#       - <prefix>_epoched_window-<L>_overlap-<O>_provenance.json

# Build a quick lookup from prefix -> {subject_ID, session_ID, num_total_samples, TR, timeseries_filepath}:
if "prefix" not in filetargets_df.columns:
    raise RuntimeError("filetargets_df must contain a 'prefix' column before export.")

prefix_metadata = filetargets_df.set_index("prefix").to_dict(orient="index")

# Set base output subdirectories:
unepoched_dir = GRAPH_METRICS_OUTPUT_DIR / "un-epoched"
unepoched_dir.mkdir(parents=True, exist_ok=True)

epoched_dir = None
if (EPOCH_LENGTH is not None) and ("EPOCHED_TIMECOURSES" in globals()) and EPOCHED_TIMECOURSES:
    epoched_root = GRAPH_METRICS_OUTPUT_DIR / "epoched"
    epoched_dir = epoched_root / f"window-{EPOCH_LENGTH}_overlap-{EPOCH_OVERLAP}"
    epoched_dir.mkdir(parents=True, exist_ok=True)


def _maybe_write_csv(df: pd.DataFrame, path: Path, index=True):
    """Write CSV if overwrite allowed or file does not exist."""
    if path.exists() and not OVERWRITE_METRICS:
        print(f"[SKIP] {path.name}: file exists and overwrite_metrics=False.")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    print(f"[SAVE] {path}")


def _maybe_write_json(payload: dict, path: Path):
    """Write JSON sidecar if overwrite allowed or file does not exist."""
    if path.exists() and not OVERWRITE_METRICS:
        print(f"[SKIP] {path.name}: JSON exists and overwrite_metrics=False.")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"[SAVE] {path}")


PAD_ROI = 3  # <-- retained for provenance / consistency if needed later


# ======================================================================
# 1) UN-EPOCHED BRANCH EXPORT
# ======================================================================

if "CENTRALITY_UNEPOCHED" in globals() and CENTRALITY_UNEPOCHED:

    print("\n[EXPORT] Un-epoched branch:")

    for prefix, centrality_df in CENTRALITY_UNEPOCHED.items():

        if prefix not in prefix_metadata:
            print(f"[WARN] Skipping un-epoched export for {prefix}: no metadata in filetargets_df.")
            continue

        meta = prefix_metadata[prefix]
        subject_id = str(meta["subject_ID"])
        session_id = str(meta["session_ID"])
        num_total_samples = int(meta["num_total_samples"])
        TR_value = float(meta["TR"])
        timeseries_filepath = str(meta["timeseries_filepath"])

        # Whole-graph summary stats (may be missing if something went wrong upstream):
        global_stats_df = GLOBAL_UNEPOCHED_STATS.get(prefix, None)
        q_unepoched_df = GRAPH_Q_UNEPOCHED.get(prefix, None)

        # -------------------------------------------------------------
        # ROI-level centrality matrix (structured)
        # -------------------------------------------------------------
        centrality_to_save = centrality_df.copy()
        centrality_to_save.index.name = f"{ATLAS_FAMILY}_ROI_label"
        centrality_path = unepoched_dir / f"{prefix}_un-epoched_centrality.csv"
        _maybe_write_csv(centrality_to_save, centrality_path, index=True)

        # -------------------------------------------------------------
        # Whole-graph metrics (single row) with subject_ID as first column
        # -------------------------------------------------------------
        if global_stats_df is not None:
            global_to_save = global_stats_df.copy()
            global_to_save.insert(0, "subject_ID", subject_id)
            global_path = unepoched_dir / f"{prefix}_un-epoched_global_metrics.csv"
            _maybe_write_csv(global_to_save, global_path, index=False)

        # -------------------------------------------------------------
        # JSON provenance for un-epoched branch
        # -------------------------------------------------------------
        n_rois_unep = centrality_df.shape[0]
        n_metrics_unep = centrality_df.shape[1]
        all_zero_cols = list(centrality_df.columns[(centrality_df == 0).all()])
        n_all_zero_cols = len(all_zero_cols)

        provenance_unep = {
            "prefix": prefix,
            "subject_id": subject_id,
            "session_id": session_id,
            "tr": TR_value,
            "num_total_samples": num_total_samples,
            "parcellation": {
                "timeseries_type": TIMESERIES_TYPE,
                "timeseries_type_token": TIMESERIES_TYPE_TOKEN,
                "atlas_family": ATLAS_FAMILY,
                "n_rois": int(NUM_ROIS),
                "network_scale": ATLAS_NETWORK_SCALE,
                "parcellation_output_dir": str(PARCELLATION_DIR),
                "timeseries_filepath": timeseries_filepath,
                "apply_gm_clip": bool(config["apply_GM_clip"]),
                "zero_threshold": float(config["zero_threshold"])},
            "compute_correlations": {
                "metrics_output_dir": str(GRAPH_METRICS_OUTPUT_DIR),
                "overwrite_metrics": OVERWRITE_METRICS,
                "correlation_type": CORRELATION_TYPE,
                "centrality_types": list(CENTRALITY_TYPES),
                "graph_metrics": list(GRAPH_METRICS),
                "absolute_weight_values": bool(ABSOLUTE_WEIGHTS),
                "epoch_length": EPOCH_LENGTH,
                "epoch_overlap": EPOCH_OVERLAP,
                "manual_TR": MANUAL_TR,
                "ensure_uniform_length": bool(ENSURE_UNIFORM_LENGTH),
                "random_seed": int(RANDOM_SEED),
                "hard_stop": bool(HARD_STOP)},
            "centrality_unepoched": {
                "n_rois": n_rois_unep,
                "n_metrics": n_metrics_unep,
                "columns": list(centrality_df.columns),
                "n_all_zero_columns": n_all_zero_cols,
                "all_zero_columns": all_zero_cols}}

        if global_stats_df is not None:
            provenance_unep["whole_graph_summaries"] = {
                "columns": list(global_stats_df.columns),
                "values": global_stats_df.iloc[0].to_dict()}

        if q_unepoched_df is not None and q_unepoched_df.shape[0] >= 1:
            provenance_unep["q_modularity"] = q_unepoched_df.iloc[0].to_dict()
        else:
            provenance_unep["q_modularity"] = None

        json_unep_path = unepoched_dir / f"{prefix}_un-epoched_provenance.json"
        _maybe_write_json(provenance_unep, json_unep_path)

else:
    print("\n[EXPORT] Un-epoched branch: no CENTRALITY_UNEPOCHED results found; skipping.")


# ======================================================================
# 2) EPOCHED BRANCH EXPORT (TIDY FORMAT)
# ======================================================================

if epoched_dir is not None and ("EPOCHED_TIMECOURSES" in globals()) and EPOCHED_TIMECOURSES:

    print("\n[EXPORT] Epoched branch (tidy outputs):")

    for prefix, timecourses_df in EPOCHED_TIMECOURSES.items():

        if prefix not in prefix_metadata:
            print(f"[WARN] Skipping epoched export for {prefix}: no metadata in filetargets_df.")
            continue

        meta = prefix_metadata[prefix]
        subject_id = str(meta["subject_ID"])
        session_id = str(meta["session_ID"])
        num_total_samples = int(meta["num_total_samples"])
        TR_value = float(meta["TR"])
        timeseries_filepath = str(meta["timeseries_filepath"])

        summaries_df = EPOCHED_SUMMARIES.get(prefix, None)

        # -------------------------------------------------------------
        # (A) Tidy timecourses + summaries CSVs
        # -------------------------------------------------------------
        timecourses_path = epoched_dir / f"{prefix}_epoched_window-{EPOCH_LENGTH}_overlap-{EPOCH_OVERLAP}_timecourses.csv"
        _maybe_write_csv(timecourses_df, timecourses_path, index=False)

        if summaries_df is not None:
            summaries_path = epoched_dir / f"{prefix}_epoched_window-{EPOCH_LENGTH}_overlap-{EPOCH_OVERLAP}_summaries.csv"
            _maybe_write_csv(summaries_df, summaries_path, index=False)

        # -------------------------------------------------------------
        # (B) JSON provenance for epoched branch
        # -------------------------------------------------------------
        # Infer number of windows from time_index in the timecourses table
        if "time_index" in timecourses_df.columns:
            window_indices = sorted(timecourses_df["time_index"].dropna().unique().astype(int).tolist())
        else:
            window_indices = []

        epoch_step = EPOCH_LENGTH - EPOCH_OVERLAP

        provenance_ep = {
            "prefix": prefix,
            "subject_id": subject_id,
            "session_id": session_id,
            "tr": TR_value,
            "num_total_samples": num_total_samples,
            "parcellation": {
                "timeseries_type": TIMESERIES_TYPE,
                "timeseries_type_token": TIMESERIES_TYPE_TOKEN,
                "atlas_family": ATLAS_FAMILY,
                "n_rois": int(NUM_ROIS),
                "network_scale": ATLAS_NETWORK_SCALE,
                "parcellation_output_dir": str(PARCELLATION_DIR),
                "timeseries_filepath": timeseries_filepath,
                "apply_gm_clip": bool(config["apply_GM_clip"]),
                "zero_threshold": float(config["zero_threshold"])},
            "compute_correlations": {
                "metrics_output_dir": str(GRAPH_METRICS_OUTPUT_DIR),
                "overwrite_metrics": OVERWRITE_METRICS,
                "correlation_type": CORRELATION_TYPE,
                "centrality_types": list(CENTRALITY_TYPES),
                "graph_metrics": list(GRAPH_METRICS),
                "absolute_weight_values": bool(ABSOLUTE_WEIGHTS),
                "epoch_length": EPOCH_LENGTH,
                "epoch_overlap": EPOCH_OVERLAP,
                "epoch_step": epoch_step,
                "manual_TR": MANUAL_TR,
                "ensure_uniform_length": bool(ENSURE_UNIFORM_LENGTH),
                "random_seed": int(RANDOM_SEED),
                "hard_stop": bool(HARD_STOP)},
            "epoched_timecourses": {
                "n_rows": int(timecourses_df.shape[0]),
                "columns": list(timecourses_df.columns),
                "window_indices": window_indices}}

        if summaries_df is not None:
            provenance_ep["epoched_summaries"] = {
                "n_rows": int(summaries_df.shape[0]),
                "columns": list(summaries_df.columns)}
        else:
            provenance_ep["epoched_summaries"] = None

        json_ep_path = epoched_dir / f"{prefix}_epoched_window-{EPOCH_LENGTH}_overlap-{EPOCH_OVERLAP}_provenance.json"
        _maybe_write_json(provenance_ep, json_ep_path)

else:
    print("\n[EXPORT] Epoched branch: no epoched results found or EPOCH_LENGTH is None; skipping.")

-----------